In [23]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == "dualtest_experiments" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "DUALTEST"))

print(PROJECT_ROOT)

c:\Users\isiva\OneDrive\Escritorio\Ingenieria de software\NLP_Proyecto_Final


In [ ]:
import pandas as pd
import time

from groq import Groq, InternalServerError, RateLimitError, APIConnectionError
from target_model import APITarget
from reference_model import ReferenceModel
from experiment_utils import prepare_records
from scoring import score_records_with_checkpoint
from sklearn.metrics import roc_auc_score

In [26]:
client = Groq(
    api_key="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"
)

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "Continue this text: The old house stood at the end of the road"}
    ],
    temperature=0,
    max_tokens=32,
)

print(response.choices[0].message.content)

The old house stood at the end of the road, its once-grand facade now worn and weathered from years of neglect. The paint had chipped and faded


In [27]:
TARGET_MODEL = "llama-3.3-70b-versatile"
TARGET_TAG = "llama33_70b_versatile"

def groq_call_with_retry(prompt, max_new_tokens=64, max_retries=8):
    wait = 5

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=TARGET_MODEL,
                messages=[
                    {
                        "role": "user",
                        "content": f"Continue the following text naturally.\n\n{prompt}",
                    }
                ],
                temperature=0,
                max_tokens=max_new_tokens,
            )
            return response.choices[0].message.content

        except (InternalServerError, RateLimitError, APIConnectionError) as e:
            print(f"Error API intento {attempt+1}/{max_retries}: {e}")
            print(f"Esperando {wait}s...")
            time.sleep(wait)
            wait = min(wait * 2, 120)

    raise RuntimeError("Groq falló después de todos los reintentos")

target = APITarget(
    call_fn=groq_call_with_retry,
    max_new_tokens=64,
)

In [28]:
reference = ReferenceModel(
    model_name="EleutherAI/pythia-410m",
    device="cpu",
)

Loading weights: 100%|██████████| 292/292 [00:00<00:00, 409.10it/s]


In [29]:
records = prepare_records(
    dataset_name="wikimia",
    n=1000,
    random_state=7,
    balance_labels=True,
    wikimia_length=256,
)

len(records), records[0].keys()

(82,
 dict_keys(['id', 'dataset', 'dataset_family', 'source_dataset', 'file_path', 'label', 'estimated_membership', 'text', 'text_hash']))

In [30]:
out_path = PROJECT_ROOT / "results" / "wikimia_llama33_70b_pythia410m_n1000_checkpoint.csv"

df = score_records_with_checkpoint(
    records=records,
    reference=reference,
    completion_fn=groq_call_with_retry,
    output_path=out_path,
    target_model_name=TARGET_MODEL,
    reference_model_name=reference.model_name,
)

df.head()

Guardados 10 / 82
Guardados 20 / 82
Guardados 30 / 82
Guardados 40 / 82
Guardados 50 / 82
Guardados 60 / 82
Guardados 70 / 82
Guardados 80 / 82


,id,dataset,label,membership,prefix,ground_truth,target_completion,run_length,p_rlb,edit_similarity,p_esb,log_p_esb,target_model,reference_model
0,WikiMIA_WikiMIA_length256_00000.txt,WikiMIA_length256,1,member,Hurricane Ana was the second tropical cyclone ...,"It rapidly consolidated, and a tropical depres...",The system gradually organized as it moved wes...,0,1.000000,0.295943,0.000000e+00,-115.081469,llama-3.3-70b-versatile,EleutherAI/pythia-410m
1,WikiMIA_WikiMIA_length256_00001.txt,WikiMIA_length256,1,member,"The 2013–2016 epidemic of Ebola virus disease,...","later, the disease spread to neighbouring Libe...","by March 2014, the outbreak had spread to neig...",0,1.000000,0.310023,1.251145e-39,-89.576759,llama-3.3-70b-versatile,EleutherAI/pythia-410m
2,WikiMIA_WikiMIA_length256_00002.txt,WikiMIA_length256,1,member,The case of Ashya King concerns a boy named As...,surgery on 24 July 2014. He received further n...,surgery at Southampton General Hospital. Howev...,1,0.000046,0.271144,0.000000e+00,-129.463719,llama-3.3-70b-versatile,EleutherAI/pythia-410m
3,WikiMIA_WikiMIA_length256_00003.txt,WikiMIA_length256,0,non_member,The 16th Houston Film Critics Society Awards w...,"films won the most awards with three each, wit...",films were major contenders in various categor...,1,0.000169,0.270886,0.000000e+00,-124.577905,llama-3.3-70b-versatile,EleutherAI/pythia-410m
4,WikiMIA_WikiMIA_length256_00004.txt,WikiMIA_length256,1,member,The Cuban thaw (Spanish: Deshielo cubano) was ...,Barack Obama and Cuban leader Raúl Castro anno...,Barack Obama and Cuban President Raúl Castro a...,5,0.000014,0.473684,0.000000e+00,-131.939888,llama-3.3-70b-versatile,EleutherAI/pythia-410m


In [31]:
df.groupby("label")[[
    "run_length",
    "edit_similarity",
    "p_rlb",
    "p_esb",
]].mean()

,run_length,edit_similarity,p_rlb,p_esb
label,,,,
0,1.870968,0.285516,0.419500,1.084876e-45
1,1.607843,0.293400,0.490206,2.763964e-37


In [ ]:
df = pd.read_csv(out_path)

print("procesados:", len(df))
print("errores:", len(df) - len(df))

print(
    "AUC run_length:",
    roc_auc_score(df["label"], df["run_length"])
)

print(
    "AUC edit_similarity:",
    roc_auc_score(df["label"], df["edit_similarity"])
)

print(
    "AUC esb log:",
    roc_auc_score(df["label"], -df["log_p_esb"])
)

procesados: 82
errores: 0
AUC run_length: 0.46679316888045536
AUC edit_similarity: 0.5471220746363061
AUC esb log: 0.3972169512966477
